In [2]:
import pandas as pd

df = pd.read_csv("dataset/synthetic_logs.csv")
df.head(3)

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert


In [3]:
filtered_df = df[df["target_label"] == "Error"]
filtered_df.head()

,timestamp,source,log_message,target_label,complexity
6,3/1/2025 19:14,ModernHR,Shard 6 replication task ended in failure,Error,bert
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert
45,5/22/2025 3:17,ThirdPartyAPI,Data replication task for shard 14 did not com...,Error,bert
51,8/19/2025 5:58,ThirdPartyAPI,Server 4 restarted without warning during data...,Error,bert
55,6/18/2025 11:21,AnalyticsEngine,Server 34 crashed unexpectedly while syncing data,Error,bert


In [4]:
filtered_df = df[df["source"] == "ModernCRM"]
filtered_df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert
15,5/1/2025 9:41,ModernCRM,Backup completed successfully.,System Notification,regex
17,2025-01-27 12:39:05,ModernCRM,nova.osapi_compute.wsgi.server [req-5f1c2027-e...,HTTP Status,bert


In [5]:
df.source.unique()

<StringArray>
[      'ModernCRM', 'AnalyticsEngine',        'ModernHR',   'BillingSystem',
   'ThirdPartyAPI',       'LegacyCRM']
Length: 6, dtype: str

In [6]:
df["target_label"].unique()

<StringArray>
[        'HTTP Status',      'Critical Error',      'Security Alert',
               'Error', 'System Notification',      'Resource Usage',
         'User Action',      'Workflow Error', 'Deprecation Warning']
Length: 9, dtype: str

In [7]:
df["target_label"].value_counts()

target_label
HTTP Status            1017
Security Alert          371
System Notification     356
Error                   177
Resource Usage          177
Critical Error          161
User Action             144
Workflow Error            4
Deprecation Warning       3
Name: count, dtype: int64

In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

# Generate embeddings for log_message column
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['log_message'].tolist())

embeddings[:2]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([[-1.02939643e-01,  3.35459560e-02, -2.20260546e-02,
         1.55099435e-03, -9.86917224e-03, -1.78956345e-01,
        -6.34410158e-02, -6.01762161e-02,  2.81108748e-02,
         5.99619932e-02, -1.72618870e-02,  1.43367937e-03,
        -1.49560049e-01,  3.15283053e-03, -5.66030554e-02,
         2.71685459e-02, -1.49890343e-02, -3.54037508e-02,
        -3.62936072e-02, -1.45410430e-02, -5.61489770e-03,
         8.75538886e-02,  4.55121212e-02,  2.50963587e-02,
         1.00187939e-02,  1.24266772e-02, -1.39923558e-01,
         7.68696293e-02,  3.14095989e-02, -4.15256852e-03,
         4.36903425e-02,  1.71250161e-02, -8.00951570e-02,
         5.74005991e-02,  1.89092141e-02,  8.55261162e-02,
         3.96399684e-02, -1.34371862e-01, -1.44364685e-03,
         3.06711951e-03,  1.76854104e-01,  4.44885204e-03,
        -1.69274919e-02,  2.24267244e-02, -4.35050242e-02,
         6.09030714e-03, -9.98171326e-03, -6.23972081e-02,
         1.07372720e-02, -6.04900671e-03, -7.14660808e-0

In [10]:
print(embeddings.shape)

(2410, 384)


In [9]:
# Cluster embeddings using DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5, metric='cosine')
clusters = dbscan.fit_predict(embeddings)

# Add cluster labels to the dataframe
df['cluster'] = clusters
df[['log_message', 'cluster']].head()

,log_message,cluster
0,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,0
1,Email service experiencing issues with sending,1
2,Unauthorized access to data was attempted,1
3,nova.osapi_compute.wsgi.server [req-4895c258-b...,0
4,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,0
